# Lecture 5 — Nonlinear least squares: Gauss–Newton and Levenberg–Marquardt

**Week 2 · Day 5 · 45 min**

> **Headline.** Curve fitting has special structure — exploit it. Gauss–Newton
> approximates the Hessian by $J^\top J$ for free; Levenberg–Marquardt damps it, and in
> doing so becomes a trust-region method.

Yesterday's Newton needed the full Hessian. Today we look at a class of problems where a
very good approximation to it comes *for free* from first derivatives alone — and where
the approximation's failure mode is repaired by the $H + \tau I$ trick you already met.

**By the end of this lecture you can:**

1. write the gradient and exact Hessian of $\tfrac12\|r(x)\|^2$;
2. say precisely which term Gauss–Newton drops and when dropping it is safe;
3. explain why $J^\top J + \lambda I$ can never break Cholesky;
4. compute a gain ratio and use it to accept, reject, and retune $\lambda$;
5. state the difference between a line search and a trust region.

**You implement this afternoon:** `LeastSquaresProblem` instances, `GaussNewton`,
`LevenbergMarquardt`.

### Pacing

Target **39 min** of core material, hard cap **45 min**. Sections marked
*(cut first)* are the ones to drop if you are running behind; everything else is
load-bearing for the labwork. **The times below already include showing and discussing
the figures** — each figure is produced by the code cell above it, so run the notebook
once before the session.


> Figure 5 is the one students remember, and it is the cheapest to show — if §3 has run
> long, cut Figure 4 rather than Figure 5.


| § | Section | min |
|---|---|---|
| 1 | The structure | 6 |
| 2 | Gauss–Newton: drop the hard term  — *Figures 1–2* | 10 |
| 3 | Levenberg–Marquardt: damp it  — *Figures 3–4* | 13 |
| 3b | Line search versus trust region  *(cut first)* | 2 |
| 4 | "Converged" does not mean "correct"  — *Figure 5* | 8 |
| 5 | Today's labs | 2 |
| | **total** | **41** |
| | **core only** | **39** |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)

# One consistent look for every figure in the lecture.
plt.rcParams.update({
    "figure.dpi": 110,
    "font.size": 9,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

# If this import fails:   pip install matplotlib

rng = np.random.default_rng(5)

---

## 1. The structure

A huge fraction of real fitting problems have this shape: you have a model
$m(t; x)$ with parameters $x$, some observations $(t_i, y_i)$, and you minimize the sum of
squared **residuals** $r_i(x) = m(t_i; x) - y_i$:

$$\min_x \; f(x) = \tfrac{1}{2}\|r(x)\|^2 = \tfrac{1}{2}\sum_{i=1}^m r_i(x)^2,
\qquad r : \mathbb{R}^n \to \mathbb{R}^m$$

Let $J(x) = \partial r/\partial x \in \mathbb{R}^{m \times n}$ be the Jacobian. Then:

$$\nabla f = J^\top r$$

$$\nabla^2 f = \underbrace{J^\top J}_{\text{first derivatives only}} \;+\; \underbrace{\sum_{i=1}^m r_i \nabla^2 r_i}_{\text{needs second derivatives}}$$

Look hard at the second term. It is a sum of the individual residual curvatures, each
**weighted by its own residual $r_i$**. So it is small when:

- the residuals are small (a good fit — and near the solution of a well-fitting model
  they are), or
- the model is nearly linear in $x$ (so $\nabla^2 r_i \approx 0$).

> **Design note.** This is why `LeastSquaresProblem` is deliberately **not** an
> `Objective`. Collapsing $r$ and $J$ into `value`/`gradient` throws away exactly the
> structure that makes today's methods work. Gauss–Newton and LM ask for `residuals` and
> `jacobian` and nothing else — interface segregation as a *modelling* decision, not
> merely a tidiness one.

---

## 2. Gauss–Newton: drop the hard term

$$\boxed{(J^\top J)\,\delta = -J^\top r}, \qquad x \leftarrow x + \delta$$

What this buys:

- **no second derivatives** — $J$ alone gives you a Hessian approximation;
- $J^\top J$ is symmetric positive **semi**-definite by construction;
- near-quadratic convergence when the dropped term really is negligible.

What it costs:

- $J^\top J$ is singular whenever $J$ is rank-deficient — and near-singular whenever two
  parameters have nearly the same effect on the model;
- with large residuals or a strongly curved model the dropped term is *not* negligible,
  the model is wrong, and the step can be far too long. **Gauss–Newton can diverge.**

Note what that first failure means statistically: a near-singular $J^\top J$ says the
data cannot distinguish two parameter directions. Same sentence in two languages, again.

In [ ]:
# The running example: exponential decay  y = a*exp(-b*t) + c
a_true, b_true, c_true = 2.5, 1.3, 0.5
t = np.linspace(0, 4, 40)
y = a_true * np.exp(-b_true * t) + c_true + 0.05 * rng.normal(size=t.size)


def residuals(x):
    a, b, c = x
    return a * np.exp(-b * t) + c - y


def jacobian(x):
    a, b, c = x
    e = np.exp(-b * t)
    return np.column_stack([e, -a * t * e, np.ones_like(t)])


def cost(x):
    r = residuals(x)
    return 0.5 * r @ r


# Check the analytic Jacobian before trusting it -- day 1 discipline, still mandatory.
def numerical_jacobian(f, x, h=1e-6):
    return np.column_stack([(f(x + h * e) - f(x - h * e)) / (2 * h)
                            for e in np.eye(len(x))])

x_test = np.array([2.0, 1.0, 0.4])
err = np.abs(jacobian(x_test) - numerical_jacobian(residuals, x_test)).max()
print(f"max |analytic J - numerical J| = {err:.2e}   {'OK' if err < 1e-6 else 'WRONG'}")

In [ ]:
def gauss_newton(x0, n_iter=30):
    x = np.asarray(x0, dtype=float)
    path = [x.copy()]
    for _ in range(n_iter):
        r, J = residuals(x), jacobian(x)
        try:
            delta = np.linalg.solve(J.T @ J, -J.T @ r)
        except np.linalg.LinAlgError:
            break
        x = x + delta
        path.append(x.copy())
        if not np.all(np.isfinite(x)):
            break
    return np.array(path)


print("GOOD START (1, 1, 1):")
p = gauss_newton([1.0, 1.0, 1.0])
for k in [0, 1, 2, 3, len(p) - 1]:
    print(f"  k={k:3d}  x = {p[k]}  cost = {cost(p[k]):.6f}")
print(f"  true      = [{a_true} {b_true} {c_true}]")

In [ ]:
# What we are actually minimizing: the total length of those vertical bars.
fig, ax = plt.subplots(1, 2, figsize=(11.5, 4.2))

tt = np.linspace(0, 4, 300)
model = lambda x, s: x[0] * np.exp(-x[1] * s) + x[2]
x_guess = np.array([1.0, 1.0, 1.0])

ax[0].plot(t, y, "o", color="k", ms=5, zorder=6, label="data $(t_i, y_i)$")
ax[0].plot(tt, model(x_guess, tt), lw=2.2, color="crimson",
           label="model at the guess $x_0$ = (1, 1, 1)")
for ti, yi in zip(t, y):
    ax[0].plot([ti, ti], [yi, model(x_guess, ti)], color="crimson", lw=1.1, alpha=0.55,
               zorder=3)
ax[0].plot([], [], color="crimson", lw=1.1, alpha=0.55, label="residuals $r_i$")
ax[0].set_xlabel("$t$"); ax[0].set_ylabel("$y$")
ax[0].set_title(f"$f(x_0) = \\frac{{1}}{{2}}\\sum_i r_i^2$ = {cost(x_guess):.3f}",
                fontsize=9)
ax[0].legend(fontsize=8)

# --- right: Gauss-Newton closing the gap, iteration by iteration.
path = gauss_newton([1.0, 1.0, 1.0])
ax[1].plot(t, y, "o", color="k", ms=5, zorder=6, label="data")
shades = ["#f4a6a6", "#e06666", "#c02020"]
for k, col in zip([0, 1, 2], shades):
    ax[1].plot(tt, model(path[k], tt), lw=1.6, color=col,
               label=f"$k$ = {k}   cost {cost(path[k]):.3f}")
ax[1].plot(tt, model(path[-1], tt), lw=2.6, color="tab:blue",
           label=f"converged  cost {cost(path[-1]):.4f}")
ax[1].set_xlabel("$t$"); ax[1].set_ylabel("$y$")
ax[1].set_title("Gauss–Newton from a good start", fontsize=9)
ax[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

**Figure 1 — what a residual is, and what we are minimizing.**

*Left:* each red bar is one residual $r_i = m(t_i; x) - y_i$ — the signed vertical gap
between the model and a data point. The objective $f(x) = \tfrac12\sum_i r_i^2$ is
(half) the total squared length of those bars, so "fitting" means sliding and bending the
red curve until the bars are collectively as short as possible.

This picture is also where the Jacobian lives: $J_{ij} = \partial r_i/\partial x_j$ says
how much bar $i$ would lengthen if you nudged parameter $j$. There are $m$ bars and $n$
parameters, which is why $J$ is $m \times n$ and tall — many more observations than
parameters.

*Right:* Gauss–Newton from a sensible start. The curve is essentially on top of the data
within three iterations. This is the case where the method is excellent, and it is worth
seeing it work before §3 shows it fall apart.

> Note that the converged cost does **not** reach zero. It settles at the noise level we
> added when building the data — the residuals that remain are the noise, not a failure of
> the optimizer. An optimizer that drove this cost to zero would be fitting the noise.

In [ ]:
print("HARD START (1, 10, 1)  -- past t = 0.5 the model is nearly constant,")
print("so a and b are barely distinguishable and J.T J is close to singular:\n")

Jh = jacobian(np.array([1.0, 10.0, 1.0]))
print(f"  cond(J.T J) at the hard start = {np.linalg.cond(Jh.T @ Jh):.3e}\n")

with np.errstate(over="ignore", invalid="ignore"):
    p = gauss_newton([1.0, 10.0, 1.0], n_iter=30)
    for k in range(min(5, len(p))):
        print(f"  k={k:3d}  cost = {cost(p[k]):.4e}   x = {p[k]}")
    print(f"\n  The very first step overshoots to a cost of 1e126. Gauss-Newton trusted a")
    print(f"  model built on a near-singular J.T J, and the step it produced was nonsense.")
    print(f"  A good fit has cost ~ {0.5 * 40 * 0.05 ** 2:.3f}.")

In [ ]:
# The same start, the two methods, on a log scale. (GN's cost leaves the page.)
def lm_track(x0, lam=1e-3, n_iter=60):
    """LM that also records lambda, rho and the accept/reject decision each step."""
    x = np.asarray(x0, dtype=float)
    hist = {"cost": [cost(x)], "lam": [], "rho": [], "accept": []}
    for _ in range(n_iter):
        r, J = residuals(x), jacobian(x)
        g = J.T @ r
        if np.linalg.norm(g) < 1e-12:
            break
        delta = np.linalg.solve(J.T @ J + lam * np.eye(len(x)), -g)
        predicted = 0.5 * delta @ (lam * delta - g)
        actual = cost(x) - cost(x + delta)
        rho = actual / predicted if predicted > 0 else -1.0
        hist["lam"].append(lam); hist["rho"].append(rho); hist["accept"].append(rho > 0)
        if rho > 0:
            x = x + delta
            lam = max(lam * max(1 / 3, 1 - (2 * rho - 1) ** 3), 1e-12)
        else:
            lam *= 10.0
        hist["cost"].append(cost(x))
    return x, hist


with np.errstate(over="ignore", invalid="ignore"):
    p_gn = gauss_newton([1.0, 10.0, 1.0], n_iter=30)
    c_gn = np.array([cost(v) for v in p_gn])
x_lm2, h2 = lm_track([1.0, 10.0, 1.0])
c_lm = np.array(h2["cost"])

fig, ax = plt.subplots(figsize=(7.8, 4.4))
ax.semilogy(np.clip(c_gn, 1e-8, 1e300), "o-", color="crimson", lw=2.0, ms=6,
            label="Gauss–Newton")
ax.semilogy(np.clip(c_lm, 1e-8, 1e300), "s-", color="tab:blue", lw=2.0, ms=5,
            label="Levenberg–Marquardt")
ax.axhline(0.5 * t.size * 0.05 ** 2, color="0.3", ls="--", lw=1.2)
ax.annotate("the noise floor — a good fit lands here",
            xy=(len(c_lm) - 1, 0.5 * t.size * 0.05 ** 2), xytext=(-4, 9),
            textcoords="offset points", fontsize=8, color="0.3", ha="right")
ax.annotate(f"one GN step: cost $10^0 \\to 10^{{{np.log10(c_gn[1]):.0f}}}$",
            xy=(1, c_gn[1]), xytext=(26, -6), textcoords="offset points", fontsize=8.5,
            color="crimson", bbox=dict(fc="white", ec="crimson", lw=0.9, alpha=0.95),
            arrowprops=dict(arrowstyle="->", color="crimson", lw=1.2))
ax.set_xlabel("iteration"); ax.set_ylabel("cost  $f(x_k)$  (log scale)")
ax.set_title("Hard start $(1, 10, 1)$: same problem, same first Jacobian", fontsize=9)
ax.legend(fontsize=9); ax.set_xlim(-1, 40)
plt.tight_layout()
plt.show()

print(f"GN  : cost went {c_gn[0]:.3f} -> {c_gn[1]:.3e} in ONE step")
print(f"LM  : final cost {c_lm[-1]:.5f} after {len(c_lm)-1} steps "
      f"({sum(1 for a in h2['accept'] if not a)} of them rejected)")

**Figure 2 — the same information, two different amounts of trust in it.**

Both methods start at $(1, 10, 1)$ and both compute the *same* first Jacobian. Gauss–Newton
believes the model it builds from it, takes the step, and the cost leaves the plot on
iteration one. Levenberg–Marquardt computes a step from the same $J$, **checks whether the
step actually helped**, finds it did not, and shrinks it.

The deep point is that Gauss–Newton is not wrong about the *direction*. It is wrong about
how far the quadratic model stays trustworthy — and it has no mechanism for finding that
out, because it never compares its prediction against reality. Every ingredient LM adds is
some form of that comparison.

---

## 3. Levenberg–Marquardt: damp it

The repair is yesterday's, applied to $J^\top J$:

$$\boxed{(J^\top J + \lambda I)\,\delta = -J^\top r}$$

Read the two limits:

- $\lambda \to 0$: this **is** Gauss–Newton — fast, and trusting the model completely;
- $\lambda \to \infty$: $\delta \to -\tfrac{1}{\lambda}J^\top r = -\tfrac{1}{\lambda}\nabla f$,
  a **small gradient-descent step** — slow, and trusting the model not at all.

So $\lambda$ interpolates continuously between "believe the quadratic model" and "just go
downhill a little". LM's job is to tune it automatically, step by step.

And note the free gift: $J^\top J \succeq 0$ always, so $J^\top J + \lambda I \succ 0$ for
any $\lambda > 0$. **Cholesky can never fail here.** Yesterday's solver is reused
unchanged, and the exception path it needs is unreachable by construction. That is a
direct payoff from having written `CholeskySolver` against an interface.

In [ ]:
# lambda as a dial between "believe the model" and "just go downhill".
x_h = np.array([1.0, 10.0, 1.0])
J_h, r_h = jacobian(x_h), residuals(x_h)
g_h = J_h.T @ r_h
lam_grid = np.logspace(-6, 4, 400)
steps = np.array([np.linalg.solve(J_h.T @ J_h + lm * np.eye(3), -g_h) for lm in lam_grid])
lens = np.linalg.norm(steps, axis=1)
angs = np.degrees(np.arccos(np.clip(
    steps @ (-g_h) / (lens * np.linalg.norm(g_h)), -1, 1)))

fig, ax = plt.subplots(1, 3, figsize=(14, 4.0))

# --- (a) the direction dial, in the (delta_a, delta_b) plane.
th = np.linspace(0, 2 * np.pi, 200)
ax[0].plot(np.cos(th), np.sin(th), color="0.8", lw=1.0)
ax[0].axhline(0, color="0.85", lw=0.8); ax[0].axvline(0, color="0.85", lw=0.8)
for lm, col in [(1e-6, "crimson"), (1e-2, "tab:orange"), (1e-1, "tab:olive"),
                (1.0, "tab:green"), (1e2, "tab:blue")]:
    d = np.linalg.solve(J_h.T @ J_h + lm * np.eye(3), -g_h)[:2]
    d = d / np.linalg.norm(d)
    lab = f"$\\lambda$ = {lm:g}" + ("  (Gauss–Newton)" if lm == 1e-6 else "")
    ax[0].plot([0, d[0]], [0, d[1]], color=col, lw=2.6, zorder=5, label=lab)
    ax[0].plot(d[0], d[1], "o", color=col, ms=6, zorder=6)
# Drawn last, on top: at large lambda the blue step lies exactly along it.
gd = (-g_h[:2]) / np.linalg.norm(-g_h[:2])
ax[0].plot([0, gd[0]], [0, gd[1]], "k--", lw=1.6, zorder=9,
           label=r"$-\nabla f$  ($\lambda \to \infty$)")
ax[0].legend(fontsize=7, loc="upper left", framealpha=0.95)
ax[0].set_aspect("equal"); ax[0].set_xlim(-1.35, 1.35); ax[0].set_ylim(-1.35, 1.6)
ax[0].set_xlabel(r"$\delta_a$ component"); ax[0].set_ylabel(r"$\delta_b$ component")
ax[0].set_title("Direction of the step, as $\\lambda$ grows", fontsize=9)
ax[0].grid(False)

# --- (b) how long the step is.
ax[1].loglog(lam_grid, lens, lw=2.4, color="tab:purple")
ax[1].axhline(lens[0], color="crimson", ls="--", lw=1.2)
ax[1].annotate(f"Gauss–Newton length = {lens[0]:.1f}", xy=(1e-6, lens[0]), xytext=(0, 7),
               textcoords="offset points", fontsize=8, color="crimson")
ax[1].loglog(lam_grid, np.linalg.norm(g_h) / lam_grid, "k:", lw=1.3,
             label=r"$\|\nabla f\|/\lambda$")
ax[1].set_xlabel(r"$\lambda$"); ax[1].set_ylabel(r"$\|\delta\|$")
ax[1].set_title(r"Large $\lambda$ $\Rightarrow$ short step", fontsize=9)
ax[1].legend(fontsize=8, loc="lower left")

# --- (c) how far the step is from the gradient direction.
ax[2].semilogx(lam_grid, angs, lw=2.4, color="tab:purple")
ax[2].axhline(angs[0], color="crimson", ls="--", lw=1.2)
ax[2].annotate(f"pure Gauss–Newton: {angs[0]:.0f}° from the gradient",
               xy=(1e-6, angs[0]), xytext=(0, -14), textcoords="offset points",
               fontsize=8, color="crimson")
ax[2].axhline(0, color="tab:blue", ls="--", lw=1.2)
ax[2].annotate("pure gradient descent: 0°", xy=(1e2, 0), xytext=(0, 8),
               textcoords="offset points", fontsize=8, color="tab:blue", ha="center")
ax[2].set_xlabel(r"$\lambda$"); ax[2].set_ylabel(r"angle between $\delta$ and $-\nabla f$")
ax[2].set_title("$\\lambda$ rotates the step, it does not only shorten it", fontsize=9)
ax[2].set_ylim(-8, 100)

plt.tight_layout()
plt.show()

print(f"at this point the Gauss-Newton step is {angs[0]:.1f} degrees from -grad f,")
print(f"and {lens[0]/lens[-1]:.3g}x longer than the step at lambda = {lam_grid[-1]:.0e}")

**Figure 3 — $\lambda$ is a dial, and it turns two things at once.**

*(a)* The **direction** of the step, drawn as a unit vector in the plane of the first two
parameters. At $\lambda = 10^{-6}$ it is the Gauss–Newton direction; as $\lambda$ grows it
swings round until it lies on top of $-\nabla f$ (dashed). Every intermediate $\lambda$
gives an intermediate direction — the dial is continuous.

*(b)* The **length**. It is flat at the Gauss–Newton length while $\lambda$ is small
compared with the eigenvalues of $J^\top J$, then falls off along $\|\nabla f\|/\lambda$
(dotted), which is the $\delta \to -\tfrac{1}{\lambda}\nabla f$ limit from the text.

*(c)* The angle between the two, which is the striking part. **At this point the
Gauss–Newton step is about 87° away from the gradient** — very nearly perpendicular to the
direction of steepest descent. That is not a bug: $J^\top J$ is severely ill-conditioned
here, and the Newton-like step correctly travels along the long axis of a stretched valley
rather than straight downhill. But it means that when the model is untrustworthy, the
Gauss–Newton step is not merely *too long* — it is pointing somewhere almost unrelated to
downhill, which is why simply shortening it would not be enough.

> **This is the difference between a line search and a trust region, in one plot.** A line
> search would search along the single red direction and find nothing good. LM changes the
> direction as well as the length — panel (a) — and that is what lets it escape.

### Choosing $\lambda$: the gain ratio

We need a principled rule. Compare what the step *promised* against what it *delivered*.

The quadratic model predicts a decrease, which for the LM step works out to

$$\text{predicted} = \tfrac{1}{2}\delta^\top(\lambda\delta - g), \qquad g = J^\top r$$

and the actual decrease is $f(x) - f(x+\delta)$. Their ratio is the **gain ratio**:

$$\rho = \frac{f(x) - f(x + \delta)}{\tfrac12 \delta^\top(\lambda\delta - g)}$$

| $\rho$ | reading | action |
|---|---|---|
| $\rho \le 0$ | the step made things *worse* — the model is lying | **reject**; increase $\lambda$ |
| small positive | model is poor but not useless | accept; increase $\lambda$ |
| $\rho \approx 1$ | model is excellent | accept; **decrease** $\lambda$ (be bolder) |

This is a feedback loop on how far the quadratic model can be trusted — and it is
self-correcting, which is why LM is the workhorse of curve fitting and why
`scipy.optimize.curve_fit` uses it.

### Line search versus trust region

$\lambda$ is the dual of a **trust-region radius** $\Delta$: a larger $\lambda$ means a
shorter, more conservative step, exactly as a smaller $\Delta$ would. The distinction
worth carrying away:

| | fixes | searches |
|---|---|---|
| **line search** (day 2) | the direction $d$ | the length $\alpha$ along it |
| **trust region** (today) | a radius $\Delta$ | direction **and** length inside the ball |

A trust region can change its mind about *where* to go, not just how far. That is what
lets LM recover from the near-singular $J^\top J$ that destroyed Gauss–Newton.

In [ ]:
def levenberg_marquardt(x0, lam=1e-3, n_iter=200, verbose=False):
    x = np.asarray(x0, dtype=float)
    n_reject = 0
    for k in range(n_iter):
        r, J = residuals(x), jacobian(x)
        g = J.T @ r
        if np.linalg.norm(g) < 1e-10:
            break
        A = J.T @ J + lam * np.eye(len(x))
        delta = np.linalg.solve(A, -g)               # always SPD: Cholesky cannot fail
        predicted = 0.5 * delta @ (lam * delta - g)
        actual = cost(x) - cost(x + delta)
        rho = actual / predicted if predicted > 0 else -1.0
        accept = rho > 0
        if verbose and k < 9:
            shown = f"{rho:8.3f}" if abs(rho) < 1e4 else f"{rho:8.1e}"
            print(f"  k={k:2d}  lam={lam:9.2e}  rho={shown}  "
                  f"{'ACCEPT' if accept else 'reject'}   cost after = "
                  f"{cost(x + delta) if accept else cost(x):.4e}")
        if accept:
            x = x + delta
            lam = max(lam * max(1 / 3, 1 - (2 * rho - 1) ** 3), 1e-12)
        else:
            n_reject += 1
            lam *= 10.0
    return x, n_reject


print("LM from the HARD start (1, 10, 1):")
x_lm, rejects = levenberg_marquardt([1.0, 10.0, 1.0], verbose=True)
print(f"\n  result      = {x_lm}")
print(f"  true        = [{a_true} {b_true} {c_true}]")
print(f"  final cost  = {cost(x_lm):.6f}")
print(f"  steps rejected along the way: {rejects}")

In [ ]:
# The feedback loop: what lambda and rho actually do over a run.
_, h = lm_track([1.0, 10.0, 1.0], n_iter=40)
ks = np.arange(len(h["lam"]))
acc = np.array(h["accept"])
rho_raw = np.array(h["rho"])
rho = np.clip(rho_raw, -1.5, 1.6)                 # clip: some steps give huge |rho|

fig, ax = plt.subplots(2, 1, figsize=(9.5, 5.6), sharex=True,
                       gridspec_kw={"height_ratios": [1, 1]})

ax[0].semilogy(ks, h["lam"], "o-", color="tab:purple", lw=1.7, ms=5)
for k in ks[~acc]:
    ax[0].axvspan(k - 0.5, k + 0.5, color="crimson", alpha=0.13, zorder=0)
ax[0].set_ylabel(r"$\lambda$")
ax[0].set_title("Red bands = the step was REJECTED and $\\lambda$ was raised tenfold",
                fontsize=9)

ax[1].axhline(0, color="0.3", lw=1.0)
ax[1].axhline(1, color="tab:green", ls="--", lw=1.2)
ax[1].annotate(r"$\rho = 1$: the model told the truth", xy=(0, 1.0), xytext=(4, 6),
               textcoords="offset points", fontsize=8, color="tab:green", ha="left")
ax[1].bar(ks[acc], rho[acc], color="tab:green", width=0.75, label=r"accepted ($\rho>0$)")
ax[1].bar(ks[~acc], rho[~acc], color="crimson", width=0.75, label=r"rejected ($\rho\leq 0$)")
ax[1].set_xlabel("iteration")
ax[1].set_ylabel(r"gain ratio $\rho$" "\n" r"(clipped to $\pm 1.5$)")
ax[1].set_ylim(-1.75, 1.95)
ax[1].legend(fontsize=8, loc="upper right", ncol=2)

plt.tight_layout()
plt.show()

print(f"rejected {int((~acc).sum())} of {len(acc)} steps; "
      f"lambda ranged {min(h['lam']):.1e} .. {max(h['lam']):.1e}")

**Figure 4 — the thermostat.**

*Top:* $\lambda$ over the run, on a log scale. It climbs steeply at the start — each red
band is a rejected step and a tenfold increase — until the step is short enough that the
model's prediction comes true. Then it falls, by orders of magnitude, as LM becomes
confident and reverts toward Gauss–Newton.

*Bottom:* the gain ratio driving it. Negative bars are steps that made things worse and
were thrown away; the parameters did not move at all on those iterations. Once $\rho$
settles near $1$ the model is predicting the actual decrease almost exactly, and LM is
free to be aggressive.

**This is a feedback controller, not a heuristic.** $\rho$ measures one thing —
*is my quadratic model telling the truth right now?* — and $\lambda$ is the single knob
that responds. Nothing here is tuned to this problem, which is why the same code fits
essentially any curve you hand it, and why `scipy.optimize.curve_fit` is LM underneath.

> Watch the first band carefully in your own run: LM's first step is the **same step
> Gauss–Newton took** in Figure 2. The difference is entirely in what happens next.

LM recovers the true parameters from the start where Gauss–Newton fell apart — and the
trace shows how: it **rejects** the first over-ambitious steps, raises $\lambda$ until the
step is short enough to be trustworthy, then lowers it again to move fast once the model
starts agreeing with reality.

---

## 4. "Converged" does not mean "correct"

One honest warning before you go and implement this.

LM is a **local** method. Nothing in this course is global. If the initial guess is in the
basin of the wrong minimum, LM will converge — confidently, quickly, with a small
gradient and a tidy `converged=True` — to the wrong answer.

The classic demonstration is fitting a sinusoid with the frequency badly initialized.
Frequency space is full of local minima, one per near-alias.

In [ ]:
# y = a * sin(omega * t + phi): many local minima in omega.
om_true, a_s, ph_true = 3.0, 1.5, 0.4
ts = np.linspace(0, 6, 120)
ys = a_s * np.sin(om_true * ts + ph_true) + 0.05 * rng.normal(size=ts.size)

def sin_cost(x):
    a, om, ph = x
    r = a * np.sin(om * ts + ph) - ys
    return 0.5 * r @ r

def sin_fit(x0, n_iter=300, lam=1e-3):
    x = np.asarray(x0, dtype=float)
    for _ in range(n_iter):
        a, om, ph = x
        s, c = np.sin(om * ts + ph), np.cos(om * ts + ph)
        r = a * s - ys
        J = np.column_stack([s, a * ts * c, a * c])
        g = J.T @ r
        d = np.linalg.solve(J.T @ J + lam * np.eye(3), -g)
        if sin_cost(x + d) < sin_cost(x):
            x, lam = x + d, max(lam / 3, 1e-12)
        else:
            lam *= 10
    return x

best = sin_cost([a_s, om_true, ph_true])
print(f"  cost at the true parameters: {best:.4f}  (this is the noise floor)\n")

for om0 in [2.8, 3.2, 4.5, 6.0, 7.5, 9.0]:
    x = sin_fit([1.0, om0, 0.0])
    tag = "good fit" if sin_cost(x) < 2 * best else "LOCAL MINIMUM -- wrong answer"
    print(f"  start omega = {om0:4.1f}  ->  omega = {x[1]:8.4f}, cost = {sin_cost(x):9.4f}   {tag}")

print(f"\n  true omega = {om_true}. Every one of these runs 'converged'.")
print("  Note the ORDER: 4.5 fails, 6.0 succeeds, 7.5 fails again. The basins are")
print("  interleaved -- 'start closer' is not a rule you can rely on.")
print("  (A recovered omega of -3 is the same curve: a*sin(-wt+p) = -a*sin(wt-p),")
print("   so the sign of omega is not identifiable. Judge by the COST, not the label.)")

In [ ]:
# The landscape that makes those runs fail: cost as a function of omega alone.
oms = np.linspace(0.2, 10.0, 600)
profile = []
for om in oms:                      # for each omega, use the BEST a and phi (profile cost)
    best = np.inf
    for ph in np.linspace(-np.pi, np.pi, 80):
        s = np.sin(om * ts + ph)
        a = (ys @ s) / (s @ s)
        best = min(best, sin_cost([a, om, ph]))
    profile.append(best)
profile = np.array(profile)

loc = [i for i in range(1, len(oms) - 1)
       if profile[i] < profile[i - 1] and profile[i] < profile[i + 1]]
gi = int(np.argmin(profile))
floor = sin_cost([a_s, om_true, ph_true])

starts = [2.8, 3.2, 4.5, 6.0, 7.5, 9.0]
outcomes = []
for om0 in starts:
    xf = sin_fit([1.0, om0, 0.0])
    outcomes.append((om0, abs(xf[1]), sin_cost(xf), sin_cost(xf) < 2 * floor))

fig, ax = plt.subplots(1, 2, figsize=(13.5, 4.5),
                       gridspec_kw={"width_ratios": [1.15, 1]})

for a_ in ax:
    a_.plot(oms, profile, lw=2.0, color="k", zorder=4)
    a_.plot(oms[loc], profile[loc], "o", color="tab:orange", ms=8, zorder=6)
    a_.plot(oms[gi], profile[gi], "*", color="gold", ms=22, mec="k", mew=1.0, zorder=7)
    a_.set_xlabel(r"frequency $\omega$")

ax[0].plot([], [], "o", color="tab:orange", ms=8, label=f"local minima ({len(loc)})")
ax[0].plot([], [], "*", color="gold", ms=16, mec="k",
           label=f"global minimum, $\\omega$ = {oms[gi]:.2f}")
ax[0].set_ylabel("best cost achievable at that $\\omega$")
ax[0].set_title("The whole landscape: one deep well, many shallow ones", fontsize=9)
ax[0].legend(fontsize=8.5, loc="lower right")

# --- right: zoom on the plateau, where the wrong answers live, with the runs drawn.
for om0, om_f, c_f, good in outcomes:
    col = "tab:green" if good else "crimson"
    y0 = np.interp(om0, oms, profile)
    ax[1].annotate("", xy=(om_f, c_f), xytext=(om0, y0),
                   arrowprops=dict(arrowstyle="->", color=col, lw=1.9, alpha=0.9,
                                   connectionstyle="arc3,rad=-0.3"))
    ax[1].plot(om0, y0, "v", color=col, ms=10, zorder=8, mec="k", mew=0.6)
    ax[1].annotate(f"{om0}", xy=(om0, y0), xytext=(0, 11), textcoords="offset points",
                   fontsize=8, color=col, ha="center", fontweight="bold")
ax[1].plot([], [], color="tab:green", lw=2.2, label="found $\\omega$ = 3  ✓")
ax[1].plot([], [], color="crimson", lw=2.2, label="stuck in a wrong dip  ✗")
ax[1].set_ylim(-4, profile.max() * 1.12)
ax[1].set_title("Where six different starting frequencies end up", fontsize=9)
ax[1].legend(fontsize=8.5, loc="lower right")

plt.tight_layout()
plt.show()

print(f"{len(loc)} local minima between omega = 0.2 and 10, and only one of them is right.")
for om0, om_f, c_f, good in outcomes:
    print(f"  start {om0:4.1f}  ->  omega = {om_f:7.4f}  cost = {c_f:8.4f}   "
          f"{'GLOBAL' if good else 'WRONG'}")

**Figure 5 — why "converged" is not a result.**

*Left:* the cost as a function of the frequency $\omega$ alone, with the amplitude and
phase set to their best values at each $\omega$. It is not a bowl. There is one deep well
at the true frequency and a row of shallow dips beside it, one per near-alias, and
**every one of those dips is a legitimate stationary point**: at the bottom of each, the
gradient is zero, the Hessian is positive definite, and any correct implementation of LM
will stop and report `converged=True`.

*Right:* the same landscape with six runs drawn on it. Green arrows reached $\omega = 3$;
red ones stopped in a wrong dip — small gradient, clean convergence flag, and a fitted
curve that is simply not the data's.

**Read the order of the arrows, because it is the real lesson.** Starting at $4.5$ fails.
Starting at $6.0$ — *further away* — succeeds. Starting at $7.5$ fails again. The basins
of attraction are interleaved, so "my initial guess was fairly close" is not a reason to
trust the answer, and neither is "it converged quickly". You cannot tell which basin you
were in by looking at anything the optimizer reports.

> **Nothing in this week can fix this.** Every method we have built is local: it uses
> $\nabla f$ and $\nabla^2 f$ *at the current point*, and no local quantity can see the
> other dips. The defences are not algorithmic — a starting value from domain knowledge,
> several restarts, and **looking at the fitted curve against the data**. This is the
> day-1 distinction between necessary and sufficient conditions, arriving with a bill
> attached.

Every one of those runs terminated happily. Some are simply wrong. **A small gradient
proves stationarity, never optimality** — which is precisely the day-1 caveat about
necessary versus sufficient conditions, arriving with consequences.

In practice: use domain knowledge for the starting point, try several starts, and look
at the fit.

---

## 5. Today's labs

| Lab | What | The point |
|---|---|---|
| 1 (45 min) | `ExpDecay`, `GaussianPeak`: `residuals` + analytic `jacobian` | validate $J$ against `numerical_jacobian` **before** optimizing |
| 2 (45 min) | `GaussNewton` | recovers parameters from a good start; **diverges** from the hard one — document it |
| 3 (70 min) | `LevenbergMarquardt` with the gain ratio | converges where GN failed; matches `scipy` on NIST StRD |

Lab 2 asks you to produce a **failure** and leave it in. That is not busywork: LM's design
is unintelligible until you have seen what it is repairing.

**Three questions for the debrief:**

1. Which term does Gauss–Newton drop, and when is dropping it dangerous?
2. Why is LM naturally compatible with Cholesky while Gauss–Newton is not?
3. How does the gain ratio decide whether to trust the model more or less?

> **Tomorrow.** Every method so far has assumed a differentiable objective. Tomorrow the
> penalty has a *kink* — and that kink is not an obstacle to work around, it is the
> feature that produces sparsity.